# India ATJ SAF techno-economic model (BioSTEAM)
Run the cells top to bottom. Files are written to `india_atj/`. See the README table in the chat for what each file does.
**Every `[PLACEHOLDER]` in `settings.py` must be replaced with a cited Indian source before you quote any result.**

In [ ]:
!pip install -q biosteam==2.54.0 chaospy
!mkdir -p india_atj

In [ ]:
%%writefile india_atj/__init__.py


In [ ]:
%%writefile india_atj/asf_model.py
"""Pure-Python chain-growth (ASF) + deactivation model. No BioSTEAM needed."""
import numpy as np
from scipy.optimize import brentq

R = 8.314  # J/mol/K

def weight_fractions(alpha, kmax=60):
    """Weight fraction of each k-mer (k ethylene units, carbon number 2k), as fraction of
    CONVERTED ethylene mass. ASF: mole fraction n_k = (1-a) a^(k-1); mass ∝ k*n_k,
    so w_k = k (1-a)^2 a^(k-1). Returns array index 0 -> k=1 (C2, i.e. butene is index 1)."""
    k = np.arange(1, kmax + 1)
    w = k * (1 - alpha) ** 2 * alpha ** (k - 1)
    return w / w.sum()

def alpha_from_T_P(T, P, beta0, dE, T_ref=393.0, P_ref=35.0):
    """alpha = 1/(1+beta); beta = termination/propagation rate ratio.
    beta = beta0 * exp(-dE/R*(1/T - 1/T_ref)) * (P_ref/P). dE = E_term - E_prop [J/mol] (>0 => hotter = more termination = shorter chains)."""
    beta = beta0 * np.exp(-dE / R * (1 / T - 1 / T_ref)) * (P_ref / P)
    return 1.0 / (1.0 + beta)

def calibrate_beta0(target_C8plus=0.63, **kw):
    """Find alpha at reference conditions so weight fraction of k>=4 (C8+) = target."""
    f = lambda a: weight_fractions(a, **kw)[3:].sum() - target_C8plus
    a = brentq(f, 0.3, 0.95)
    return a, (1 - a) / a            # alpha_ref, beta0

def cycle_average_activity(kd, tau_days):
    """First-order deactivation a(t)=exp(-kd t); mean activity over a run of tau days."""
    x = kd * tau_days
    return (1 - np.exp(-x)) / x if x > 1e-12 else 1.0

def conversion(X0, kd, tau_days):
    return X0 * cycle_average_activity(kd, tau_days)

if __name__ == "__main__":
    a, b0 = calibrate_beta0(0.63)
    print(f"alpha_ref={a:.3f} beta0={b0:.3f}")
    w = weight_fractions(a)
    print("C4-C6 :", w[1:3].sum().round(3), "C8-C16:", w[3:8].sum().round(3), "C18+:", w[8:].sum().round(3))


In [ ]:
%%writefile india_atj/settings.py
"""
settings.py  --  ALL inputs of the India ATJ model live here (nothing is hard-coded elsewhere).

Convention: money inputs are in INR where Indian data exist; the BioSTEAM model itself works in USD,
so `fx` converts. Every number carries a source tag:
  [VERIFIED-26]  = checked by web search on 19 Oct 2026 while writing this file (source URL given)
  [REPO]         = carried over unchanged from Wadgama & Shi 2026 / github.com/H-Wadgama/belt (US basis)
  [PLACEHOLDER]  = my estimate, NOT verified -> you must replace it with a cited Indian source
"""
import copy

S = dict(
    # ------------------------------------------------------------------ macro / finance
    fx=95.5,                      # INR per USD. [VERIFIED-26] spot ~95.8 (Jul 2026) to ~95.95 (16 Sep 2026): forecasts.org, longforecast.com. Re-check on the day you run.
    CEPCI=800.8,                  # [REPO] 2023 Chemical Engineering Plant Cost Index used by the paper (chemengonline.com/pci-home)
    location_factor=0.90,         # [PLACEHOLDER] India/US installed-cost ratio. Replace with a cited value (e.g. Peters&Timmerhaus-type location indices or an Indian EPC quote)
    IRR=0.10,                     # [REPO] after-tax return used by the paper to define MJSP (NPV=0)
    income_tax=0.2517,            # [PLACEHOLDER-check] 25.17% = 22% base + 10% surcharge + 4% cess (Income-tax Act s.115BAA). Verify on incometaxindia.gov.in
    finance_fraction=0.60,        # [REPO] 60% debt
    finance_interest=0.095,       # [PLACEHOLDER] Indian term-loan rate; paper uses 8% (US). Verify against SBI/IREDA project-loan rates
    finance_years=10,             # [REPO]
    duration=(2027, 2057),        # 30-yr life [REPO]; start year is my assumption
    operating_days=330,           # [REPO] (= 90.4% on-stream)
    construction_schedule=(0.08, 0.60, 0.32),   # [REPO]
    WC_over_FCI=0.05,             # [REPO]
    wdv_rate=0.15,                # [PLACEHOLDER-check] Indian IT-Act written-down-value rate for plant & machinery (15%). Verify (Appendix I, IT Rules). Replaces US MACRS-7.
    # ------------------------------------------------------------------ plant size
    ethanol_KLPD=200.0,           # design basis, kilolitres/day of anhydrous ethanol. Typical large Indian distillery range 100-300 KLPD (ISMA/ MoPNG). Your scale sweep varies this.
    ethanol_purity_wt=0.995,      # [REPO] feed_parameters['purity']
    ethanol_density=789.0,        # kg/m3, [REPO] (Humbird 2011)
    # ------------------------------------------------------------------ ethanol feedstock price
    # Administered ex-mill prices, Rs/litre, ESY 2025-26 (1 Nov 2025 - 31 Oct 2026). [VERIFIED-26]
    #  sources: dataful.in/datasets/21605 (MoPNG series); bioenergytimes.com (Jul 2026, MoS Petroleum written reply);
    #           iamrenew.com / bioenergytimes.com (FCI rice Rs 60.32).
    ethanol_route='B_heavy',
    ethanol_exmill_INR_L={'C_heavy': 57.97, 'B_heavy': 60.73, 'cane_juice': 65.61,
                          'FCI_rice': 60.32, 'damaged_grain': 64.00, 'maize': 71.86,
                          '2G_cellulosic': 80.0},   # 2G value = [PLACEHOLDER], no administered price verified; widen its uncertainty
    ethanol_transport_INR_L=1.2,  # DERIVED from [VERIFIED-26]: delivered 71.18 (IOCL) - 66.61*1.05 (ex-mill + 5% GST) = ~1.24 Rs/L. GST assumed creditable (excluded).
    # ------------------------------------------------------------------ hydrogen
    h2_route='green_offsite',
    # delivered H2 cost to the plant incl. compression/delivery, INR/kg.  [PLACEHOLDER - NOT VERIFIED] -> look at
    # NGHM/SIGHT tender results (mnre.gov.in), SECI green-H2 tenders, refinery SMR cost (PPAC / company reports).
    h2_INR_kg={'green_offsite': 450.0, 'green_onsite': 420.0, 'grey_smr_refinery': 200.0, 'blue_smr_ccs': 260.0},
    h2_storage_days=7,            # [REPO]
    h2_ratio=3.0,                 # mol H2 : mol olefin at hydrogenator inlet [REPO]
    h2_recovery=0.85,             # PSA recovery [REPO]
    # ------------------------------------------------------------------ co-products & utilities
    # Co-product prices: [REPO] US basis (BASF 2023 key commodities; AFDC 2023), converted to Rs for transparency.
    # [PLACEHOLDER] replace by Indian naphtha / HSD ex-refinery prices from PPAC (ppac.gov.in -> Price Build-up / Import-Export).
    rn_USD_kg=0.71, rd_USD_kg=1.888,
    elec_INR_kWh=7.5,             # [PLACEHOLDER] HT industrial tariff, state-dependent (CEA / state SERC orders). paper: $0.0782/kWh
    ng_INR_kg=45.0,               # [PLACEHOLDER] furnace natural gas, Rs/kg (~Rs 900/MMBtu-equivalent check!). Replace from PPAC/GAIL.
    wastewater_USD_kg=1.85e-3,    # [REPO] Humbird-style WWT cost per kg wastewater
    labor_USD_yr=1.2e6,           # [PLACEHOLDER] paper-style 2.5e6 USD scaled down for Indian wages; replace with cited wages (e.g. Periodic Labour Force Survey / plant HR data)
    # ------------------------------------------------------------------ catalysts ($/kg, global/imported)  [REPO]
    cat_price_USD_kg={'dehyd': 36.81, 'olig': 158.4, 'hydgn': 59.12},   # Zhang&Yu 2013 / Tao 2017 / Davis 2015 via repo's etj_settings.py
    cat_life_yr={'dehyd': 2.0, 'olig': 1.0, 'hydgn': 3.0},              # [REPO]
    # ------------------------------------------------------------------ reactors
    dehyd=dict(T=754.0, P=10.63e5, X=0.995, whsv=0.3),   # [REPO]; Syndol alumina catalyst, Zhang & Yu 2013 (Ind. Eng. Chem. Res. 52, 9505)
    olig=dict(T=393.15, P=35e5, X0=0.993, whsv=1.5,       # [REPO]; Heveling et al. 1998 Appl. Catal. A 173, 1-9
              c8plus_ref=0.63,                              # Heveling: 63 wt% C8+ at 393 K / 35 bar
              dE=15e3,                                      # [PLACEHOLDER] J/mol, E_term - E_prop; FIT TO DATA (Heveling 1998, Eagan 2019 Nat Rev Chem 3:223)
              kd=0.002, cycle_days=60.0),                   # [PLACEHOLDER] deactivation 1/day, run length; no literature value found - treat as uncertain
    hydgn=dict(T=623.15, P=35e5, X=1.0, whsv=3.0),        # [REPO]; Tao et al. 2017 Green Chem 19:1082
    # ------------------------------------------------------------------ policy / benchmark (used only in analysis.py)
    fossil_atf_INR_L=None,        # [PLACEHOLDER] set from IOCL/PPAC monthly ATF price (Rs/kL) before using the competitiveness plots
)

def get():
    """Return a fresh deep copy so scenario runs never contaminate each other."""
    return copy.deepcopy(S)

# ---- unit helpers --------------------------------------------------------
def ethanol_kg_hr(s):
    return s['ethanol_KLPD'] * s['ethanol_density'] / 24.0          # KL/day -> kg/hr  (1 KL = 1 m3)

def ethanol_price_USD_kg(s):
    inr_l = s['ethanol_exmill_INR_L'][s['ethanol_route']] + s['ethanol_transport_INR_L']
    return inr_l / s['fx'] / (s['ethanol_density'] / 1000.0)          # Rs/L -> USD/kg

def h2_price_USD_kg(s):
    return s['h2_INR_kg'][s['h2_route']] / s['fx']

def elec_USD_kWh(s): return s['elec_INR_kWh'] / s['fx']
def ng_USD_kg(s):    return s['ng_INR_kg'] / s['fx']


In [ ]:
%%writefile india_atj/chemicals.py
"""
chemicals.py -- the molecule list (thermosteam Chemicals).
Depends on: nothing local. Used by: units.py, system.py.

Change vs the paper's repo: instead of 4 lumps (C4, C6, C10, C18) we carry EVERY ASF chain length
C4, C6, ... C16 plus one C18+ lump, so the fractionation columns cut at real carbon-number boundaries
(RN = C4-C6, SAF = C8-C16, RD = C18+). This also removes the repo's decane-vs-dodecane surrogate mismatch.
"""
import thermosteam as tmo

OLEFINS = {2: '1-Butene', 3: '1-Hexene', 4: '1-Octene', 5: '1-Decene', 6: '1-Dodecene',
           7: '1-Tetradecene', 8: '1-Hexadecene', 9: '1-Octadecene'}          # key k = number of C2 units
PARAFFINS = {2: 'Butane', 3: 'Hexane', 4: 'Octane', 5: 'Decane', 6: 'Dodecane',
             7: 'Tetradecane', 8: 'Hexadecane', 9: 'Octadecane'}
RN_K, SAF_K, RD_K = (2, 3), (4, 5, 6, 7, 8), (9,)

def create_chemicals():
    ids = ['Water', 'Ethanol', 'Ethylene', 'Hydrogen'] + list(OLEFINS.values()) + list(PARAFFINS.values())
    chems = [tmo.Chemical(i) for i in ids]
    # Solid catalysts: no phase equilibrium needed, only a molar mass so mass flows can be tracked.
    # MW placeholders; Syndol ~ alumina-based (Zhang & Yu 2013), Ni/SiO2-Al2O3 (Heveling 1998), CoMo/Al2O3 (Tao 2017)
    for name, mw in [('Syndol', 102.0), ('Nickel_SiAl', 100.0), ('CobaltMolybdenum', 150.0)]:
        chems.append(tmo.Chemical(name, search_db=False, default=True, phase='s', MW=mw))
    cs = tmo.Chemicals(chems)
    cs.compile()
    return cs


In [ ]:
%%writefile india_atj/units.py
"""
units.py -- custom BioSTEAM unit classes.  Depends on: asf_model.py, chemicals.py.  Used by: system.py.

 CatalystReactor      generic catalytic fixed bed: catalyst mass = feed/WHSV, pressure-vessel cost, catalyst makeup stream
   DehydrationReactor ethanol -> ethylene + water (adiabatic, fixed 99.5% conversion)
   ASFOligomerizer    ethylene -> C4..C18+ olefins, selectivity from ASF chain-growth model, deactivation-averaged conversion
   Hydrogenator       olefins + H2 -> paraffins (adiabatic, 100%)
 StorageTank          generic power-law tank/vessel cost (ethanol, H2, products)
"""
import numpy as np
import biosteam as bst
from .asf_model import weight_fractions, alpha_from_T_P, calibrate_beta0, conversion
from .chemicals import OLEFINS, PARAFFINS

def vessel_purchase_cost(V_m3, P_Pa, LD=3.0, sigma=15000., E=0.85):
    """Vertical carbon-steel pressure vessel, Seider et al. 'Product & Process Design Principles' 4th ed. (Wiley 2017)
    correlations (CE=500 basis) -- coefficients typed from memory: VERIFY against the book before publishing."""
    D = (4 * V_m3 / (np.pi * LD)) ** (1 / 3); L = LD * D
    Dft, Lft = D * 3.28084, L * 3.28084
    Pg = max(P_Pa / 6894.76 - 14.7, 15.)
    tp = max(Pg * Dft * 12 / (2 * sigma * E - 1.2 * Pg), 0.25) + 0.125            # wall thickness [in]
    W = np.pi * (Dft + tp / 12) * (Lft + 0.8 * Dft) * (tp / 12) * 490.             # shell weight [lb]
    Cv = np.exp(7.0132 + 0.18255 * np.log(W) + 0.02297 * np.log(W) ** 2)
    Cpl = 410 * Dft ** 0.7396 * Lft ** 0.70684
    return (Cv + Cpl) * bst.CE / 500.

class CatalystReactor(bst.Unit):
    """ins = (feed, catalyst_makeup).  catalyst makeup is a purchased feed stream whose flow is
    inventory / lifetime / 8760 (this is the 'replacement cost' line of the TEA)."""
    _N_ins, _N_outs = 2, 1
    _F_BM_default = {'Reactor': 4.16, 'Initial catalyst': 1.0}
    _units = {'Catalyst mass': 'kg', 'Reactor volume': 'm3'}
    cat_id = None

    def __init__(self, ID='', ins=None, outs=(), thermo=None, T=None, P=None, whsv=1., cat_life_yr=1.,
                 bed_density=700., isothermal=False):
        super().__init__(ID, ins, outs, thermo)
        self.T, self.P, self.whsv, self.cat_life_yr = T, P, whsv, cat_life_yr
        self.bed_density, self.isothermal = bed_density, isothermal

    def _react(self, feed, out):  raise NotImplementedError

    def _run(self):
        feed, cat = self.ins; out, = self.outs
        out.mol[:] = 0; out.mol[:] = feed.mol            # aggregate moles of all phases
        out.phase = 'g'
        self._react(feed, out)
        out.T = self.T if (self.isothermal or self.T) else feed.T
        out.P = self.P
        m_cat = feed.F_mass / self.whsv
        cat.empty(); cat.imass[self.cat_id] = m_cat / self.cat_life_yr / 8760.   # kg/hr of makeup
        self.m_cat = m_cat

    def _design(self):
        feed = self.ins[0]
        self.design_results['Catalyst mass'] = self.m_cat
        self.design_results['Reactor volume'] = self.m_cat / self.bed_density / 0.5   # bed fraction 0.5
        duty = self.outs[0].H - feed.H
        self.add_heat_utility(duty, self.T or feed.T)

    def _cost(self):
        self.baseline_purchase_costs['Reactor'] = vessel_purchase_cost(
            self.design_results['Reactor volume'], self.P)
        # initial catalyst charge is capital
        price = self.ins[1].price or 0.
        self.baseline_purchase_costs['Initial catalyst'] = price * self.m_cat

class DehydrationReactor(CatalystReactor):
    cat_id = 'Syndol'
    def __init__(self, *a, X=0.995, **k):
        super().__init__(*a, **k); self.X = X
    def _react(self, feed, out):
        n = feed.imol['Ethanol'] * self.X
        out.imol['Ethanol'] -= n; out.imol['Ethylene'] += n; out.imol['Water'] += n

class ASFOligomerizer(CatalystReactor):
    cat_id = 'Nickel_SiAl'
    def __init__(self, *a, X0=0.993, c8plus_ref=0.63, dE=15e3, kd=0.002, cycle_days=60., **k):
        super().__init__(*a, isothermal=True, **k)
        self.X0, self.dE, self.kd, self.cycle_days, self.c8plus_ref = X0, dE, kd, cycle_days, c8plus_ref
        self.beta0 = calibrate_beta0(c8plus_ref)[1]      # calibrates alpha to the Heveling 63 wt% C8+ point
    @property
    def alpha(self): return float(alpha_from_T_P(self.T, self.P / 1e5, self.beta0, self.dE))
    def _react(self, feed, out):
        X = conversion(self.X0, self.kd, self.cycle_days)    # cycle-average conversion after deactivation
        w = weight_fractions(self.alpha); w = np.r_[w[:8], w[8:].sum()]   # lump k>=9 into C18
        n0 = feed.imol['Ethylene']
        self.X_avg, self.w = X, w
        for k in range(2, 10):
            nc = n0 * X * w[k - 1]                      # mol ethylene -> k-mer (weight fraction == mole-of-C2 fraction)
            out.imol['Ethylene'] -= nc
            out.imol[OLEFINS[k]] += nc / k

class Hydrogenator(CatalystReactor):
    cat_id = 'CobaltMolybdenum'
    def _react(self, feed, out):
        for k in range(2, 10):
            n = feed.imol[OLEFINS[k]] * 1.0
            out.imol[OLEFINS[k]] -= n; out.imol[PARAFFINS[k]] += n; out.imol['Hydrogen'] -= n

class StorageTank(bst.Unit):
    """Power-law cost  C = C0 * (CE/CE0) * (capacity/cap0)**exp  * n_units.  Capacity in gal (liquids) or kg (H2).
    Sources (carried from the paper's repo atj_bst_units.py -- VERIFY the originals):
      ethanol : Humbird et al. 2011 NREL/TP-5100-47764 (C0=1.7*1.34e6 USD, 750,000 gal, CE0=521.9, exp 0.7)
      product : Dutta et al. 2015 PNNL-23823 (C0=1.5534e6 USD, 500,000 gal, CE0=567.3, exp 0.7)
      H2      : Amos 1998 NREL/TP-570-25106 (pressurised vessels; repo uses 600*500 USD at 500/2.2 kg, exp 0.75,
                max 1300 kg/vessel). CORRECTION: repo applied NO CEPCI update to H2 tanks; we scale from CE0=389.5 (1998) -- check value."""
    _N_ins, _N_outs = 1, 1
    _F_BM_default = {'Tank': 1.0}
    _units = {'Capacity': 'gal or kg'}
    def __init__(self, ID='', ins=None, outs=(), thermo=None, kind='liquid', days=7, C0=1.5534e6,
                 cap0=500000., CE0=567.3, exp=0.7, max_unit=None):
        super().__init__(ID, ins, outs, thermo)
        self.kind, self.days, self.C0, self.cap0, self.CE0, self.exp, self.max_unit = kind, days, C0, cap0, CE0, exp, max_unit
    def _run(self): self.outs[0].copy_like(self.ins[0])
    def _design(self):
        f = self.ins[0]
        cap = f.F_mass * self.days * 24 if self.kind == 'mass' else f.F_vol * 264.172 * self.days * 24
        self.design_results['Capacity'] = cap
        self.n = int(np.ceil(cap / self.max_unit)) if self.max_unit else 1
    def _cost(self):
        cap = self.design_results['Capacity'] / self.n
        one = self.C0 * (bst.CE / self.CE0) * (cap / self.cap0) ** self.exp
        self.baseline_purchase_costs['Tank'] = one * self.n


In [ ]:
%%writefile india_atj/tea.py
"""
tea.py -- discounted-cash-flow economics with Indian finance. Depends on: settings.py (numbers only).
Used by: system.py (builds it), analysis.py (solves MJSP).

Same method as the paper (BioSTEAM TEA, NPV=0 at IRR 10% -> minimum selling price) with India changes:
  * income tax 25.17%, loan 9.5%  (settings.py)
  * depreciation = Indian written-down-value (WDV) schedule instead of US MACRS-7
  * location_factor on installed cost
Capital-cost build-up fractions are the Humbird et al. 2011 (NREL/TP-5100-47764) values the paper's TEA also uses.
"""
import numpy as np
import biosteam as bst

class IndiaTEA(bst.TEA):
    def __init__(self, system, s, **kw):
        n = s['duration'][1] - s['duration'][0]
        r = s['wdv_rate']
        dep = np.array([r * (1 - r) ** t for t in range(n)])          # WDV: d_t = r(1-r)^(t-1); remainder stays undepreciated (simplification)
        dep = dep / dep.sum()                                          # normalise so 100% is written off over life (see notes)
        super().__init__(system, IRR=s['IRR'], duration=s['duration'], depreciation=dep,
                         income_tax=s['income_tax'], operating_days=s['operating_days'], lang_factor=None,
                         construction_schedule=s['construction_schedule'], startup_months=3,
                         startup_FOCfrac=1., startup_VOCfrac=0.75, startup_salesfrac=0.5,
                         WC_over_FCI=s['WC_over_FCI'], finance_interest=s['finance_interest'],
                         finance_years=s['finance_years'], finance_fraction=s['finance_fraction'])
        self.s = s
        self.labor_cost = s['labor_USD_yr']; self.labor_burden = 0.90
        self.property_insurance, self.maintenance = 0.007, 0.03        # Humbird 2011 / paper's cellulosic TEA

    def _DPI(self, installed_equipment_cost):   # + warehouse 4%, site development 9%, additional piping 4.5%
        return installed_equipment_cost * (1 + 0.04 + 0.09 + 0.045)
    def _TDC(self, DPI):                         # direct costs (no further adders)
        return DPI
    def _FCI(self, TDC):                         # indirect: prorateable 10, field 10, construction 20, contingency 10, other 10
        return TDC * (1 + 0.10 + 0.10 + 0.20 + 0.10 + 0.10) * self.s['location_factor']
    def _FOC(self, FCI):
        return FCI * (self.property_insurance + self.maintenance) + self.labor_cost * (1 + self.labor_burden)


In [ ]:
%%writefile india_atj/system.py
"""
system.py -- builds the flowsheet. Depends on: settings.py, chemicals.py, units.py, tea.py.
Used by: analysis.py, uncertainty.py.   Public API:  build(s) -> (system, tea, F)   and   apply_prices(F, s).

Flow:  ethanol -> T101 -> pump -> heat -> [mix w/ recycle] -> heat -> R201 dehydration -> split(70% recycle)
       -> flash (drop water) -> compress -> D201 (water) -> compress -> D202 (trace ethanol) -> cool
       -> [mix w/ ethylene recycle] -> R202 ASF oligomerizer -> split (unreacted C2H4 recycle)
       -> mix w/ H2 -> heat -> R203 hydrogenation -> cool -> flash -> PSA split (H2 recycle)
       -> D301 (RN | rest) -> D302 (SAF | RD) -> coolers -> product tanks
"""
import biosteam as bst
from . import settings as cfg
from .chemicals import create_chemicals, OLEFINS, PARAFFINS, RN_K, SAF_K, RD_K
from .units import DehydrationReactor, ASFOligomerizer, Hydrogenator, StorageTank
from .tea import IndiaTEA

def apply_prices(F, s):
    """THE single place where prices are written onto streams/utilities. Uncertainty setters call this again
    after changing `s` (CORRECTION: the paper's notebook changed a dict *after* the stream price was set,
    so hydrogen-price uncertainty probably never reached the model)."""
    F.stream.Ethanol_In.price = cfg.ethanol_price_USD_kg(s)
    F.stream.Hydrogen_In.price = cfg.h2_price_USD_kg(s)
    F.stream.RN.price = s['rn_USD_kg']          # CORRECTION: naphtha setter must touch RN (paper's notebook wrote to RD)
    F.stream.RD.price = s['rd_USD_kg']
    F.stream.Wastewater.price = -s['wastewater_USD_kg']     # disposal cost = negative revenue
    for key, sid in (('dehyd', 'Cat_dehyd'), ('olig', 'Cat_olig'), ('hydgn', 'Cat_hydgn')):
        F.stream[sid].price = s['cat_price_USD_kg'][key]
    bst.PowerUtility.price = cfg.elec_USD_kWh(s)
    bst.HeatUtility.get_agent('natural_gas').regeneration_price = cfg.ng_USD_kg(s) * 16.04   # USD/kmol CH4

def build(s=None, ID='india_atj'):
    s = s or cfg.get()
    bst.settings.CEPCI = s['CEPCI']
    bst.main_flowsheet.set_flowsheet(ID)
    F = bst.main_flowsheet
    bst.settings.set_thermo(create_chemicals())
    eth = cfg.ethanol_kg_hr(s)
    feed = bst.Stream('Ethanol_In', Ethanol=eth, Water=eth * (1 - s['ethanol_purity_wt']) / s['ethanol_purity_wt'],
                      units='kg/hr', T=293.15, P=101325, phase='l')
    d, o, h = s['dehyd'], s['olig'], s['hydgn']
    dehyd_rec = bst.MultiStream('dehyd_recycle', phases=('g', 'l'))
    eth_rec = bst.MultiStream('ethylene_recycle', phases=('g', 'l'))
    h2_rec = bst.Stream('h2_recycle', P=3e6, phase='g')

    T101 = StorageTank('T101', feed, kind='liquid', days=7, C0=1.7 * 1.34e6, cap0=750000., CE0=521.9, exp=0.7)
    P201 = bst.Pump('P201', T101-0, P=1.373e6)
    H201 = bst.HXutility('H201', P201-0, T=500., rigorous=True)
    M201 = bst.Mixer('M201', (H201-0, dehyd_rec), rigorous=True)
    H202 = bst.HXutility('H202', M201-0, T=d['T'], rigorous=True)
    R201 = DehydrationReactor('R201', (H202-0, bst.Stream('Cat_dehyd')), T=d['T'], P=d['P'], whsv=d['whsv'],
                              cat_life_yr=s['cat_life_yr']['dehyd'], X=d['X'])
    S201 = bst.Splitter('S201', R201-0, outs=('to_flash', dehyd_rec), split=0.3)
    T201 = bst.Flash('T201', S201-0, T=420., P=d['P'])
    K201 = bst.IsentropicCompressor('K201', T201-0, P=2e6, eta=0.72)
    D201 = bst.BinaryDistillation('D201', K201-0, LHK=('Ethylene', 'Water'), y_top=0.999, x_bot=0.001, k=2, P=2e6)
    K202 = bst.IsentropicCompressor('K202', D201-0, P=o['P'], eta=0.72)
    D202 = bst.BinaryDistillation('D202', K202-0, LHK=('Ethylene', 'Ethanol'), y_top=0.9999, x_bot=0.0001, k=2, P=o['P'])
    H203 = bst.HXutility('H203', D202-0, T=o['T'], rigorous=True)
    M202 = bst.Mixer('M202', (H203-0, eth_rec), rigorous=True)
    R202 = ASFOligomerizer('R202', (M202-0, bst.Stream('Cat_olig')), T=o['T'], P=o['P'], whsv=o['whsv'],
                           cat_life_yr=s['cat_life_yr']['olig'], X0=o['X0'], c8plus_ref=o['c8plus_ref'],
                           dE=o['dE'], kd=o['kd'], cycle_days=o['cycle_days'])
    S202 = bst.Splitter('S202', R202-0, outs=(eth_rec, 'olefins'), split={'Ethylene': 1.0})

    h2_in = bst.Stream('Hydrogen_In', P=3e6, phase='g')
    T102 = StorageTank('T102', h2_in, kind='mass', days=s['h2_storage_days'], C0=600 * 500, cap0=500 / 2.2,
                       CE0=389.5, exp=0.75, max_unit=1300.)
    @T102.add_specification(run=True)
    def h2_flow():     # fresh H2 = ratio*olefin_mol - recycled H2
        n_ol = sum(R202.outs[0].imol[OLEFINS[k]] for k in OLEFINS)
        h2_in.imol['Hydrogen'] = max(s['h2_ratio'] * n_ol - h2_rec.imol['Hydrogen'], 0.)
    M203 = bst.Mixer('M203', (T102-0, S202-1, h2_rec), rigorous=True)
    H204 = bst.HXutility('H204', M203-0, T=h['T'], rigorous=True)
    R203 = Hydrogenator('R203', (H204-0, bst.Stream('Cat_hydgn')), T=h['T'], P=h['P'], whsv=h['whsv'],
                        cat_life_yr=s['cat_life_yr']['hydgn'])
    H205 = bst.HXutility('H205', R203-0, T=250., rigorous=True)
    T202 = bst.Flash('T202', H205-0, T=250., P=5e5)
    S203 = bst.Splitter('S203', T202-0, outs=(h2_rec, 'offgas'), split={'Hydrogen': s['h2_recovery']})
    D301 = bst.BinaryDistillation('D301', T202-1, LHK=(PARAFFINS[3], PARAFFINS[4]), y_top=0.99, x_bot=0.01, k=2, is_divided=True)
    D302 = bst.BinaryDistillation('D302', D301-1, LHK=(PARAFFINS[8], PARAFFINS[9]), y_top=0.99, x_bot=0.01, k=2, is_divided=True)
    H301 = bst.HXutility('H301', D301-0, V=0, rigorous=True)
    H302 = bst.HXutility('H302', D302-0, T=288.15, rigorous=True)
    H303 = bst.HXutility('H303', D302-1, T=288.15, rigorous=True)
    for u in (H302, H303):
        @u.add_specification(run=False)
        def fix_phase(u=u):
            u._run(); u._design(); u._cost(); u.outs[0].phase = 'l'
    T501 = StorageTank('T501', H301-0, outs='RN',  kind='liquid', days=14)
    T502 = StorageTank('T502', H302-0, outs='SAF', kind='liquid', days=14)
    T503 = StorageTank('T503', H303-0, outs='RD',  kind='liquid', days=14)
    WW = bst.Mixer('M601', (T201-1, D201-1, D202-1), outs='Wastewater')

    path = (T101, P201, H201, M201, H202, R201, S201, T201, K201, D201, K202, D202, H203, M202, R202, S202,
            T102, M203, H204, R203, H205, T202, S203, D301, D302, H301, H302, H303, T501, T502, T503, WW)
    sys = bst.System(ID, path=path, recycle=(dehyd_rec, eth_rec, h2_rec))
    apply_prices(F, s)
    tea = IndiaTEA(sys, s)
    return sys, tea, F


In [ ]:
%%writefile india_atj/analysis.py
"""
analysis.py -- everything you RUN. Depends on: settings.py, system.py (and asf_model.py indirectly).
Each function builds a fresh flowsheet from a (possibly modified) copy of settings, simulates, and returns numbers.
"""
import itertools, warnings
import numpy as np, pandas as pd
import biosteam as bst
from . import settings as cfg
from .system import build

warnings.filterwarnings('ignore')
_n = itertools.count(1)
GAL = 264.172        # gal per m3

def run(s=None, **overrides):
    """Build + simulate. overrides use dotted keys, e.g. run(**{'olig.T': 403.15, 'ethanol_route': 'maize'})."""
    s = s or cfg.get()
    for k, v in overrides.items():
        if '.' in k:
            a, b = k.split('.'); s[a][b] = v
        else:
            s[k] = v
    sys, tea, F = build(s, ID=f'run{next(_n)}')
    sys.simulate()
    return sys, tea, F, s

def results(sys, tea, F, s):
    """One-row summary. Units: USD/gal and INR/L for MJSP; H2 intensity computed correctly (kg H2 per tonne of fuel)."""
    st = F.stream; hrs = tea.operating_hours
    p = tea.solve_price(st.SAF)                                    # USD/kg  (NPV = 0 at IRR)
    saf_gal = st.SAF.F_vol * GAL * hrs
    fuel_kg = st.SAF.F_mass + st.RN.F_mass + st.RD.F_mass
    ol = F.unit.R202
    return dict(MJSP_USD_gal=p * st.SAF.rho / GAL, MJSP_INR_L=p * st.SAF.rho / 1000 * s['fx'],
                SAF_MMgal_yr=saf_gal / 1e6, SAF_frac_of_fuel=st.SAF.F_mass / fuel_kg,
                RN_MMgal_yr=st.RN.F_vol * GAL * hrs / 1e6, RD_MMgal_yr=st.RD.F_vol * GAL * hrs / 1e6,
                alpha=ol.alpha, olig_X_avg=ol.X_avg,
                H2_kg_hr=st.Hydrogen_In.F_mass, H2_kg_per_tonne_fuel=st.Hydrogen_In.F_mass / fuel_kg * 1000,
                installed_MM_USD=tea.installed_equipment_cost / 1e6, FCI_MM_USD=tea.FCI / 1e6,
                FOC_MM_USD_yr=tea.FOC / 1e6, VOC_MM_USD_yr=tea.VOC / 1e6, sales_MM_USD_yr=tea.sales / 1e6,
                ethanol_cost_share=st.Ethanol_In.cost * hrs / (tea.VOC + tea.FOC),
                mass_balance_pct=(sum(x.F_mass for x in sys.feeds) - sum(x.F_mass for x in sys.products))
                                  / sum(x.F_mass for x in sys.feeds) * 100)

def baseline():
    return pd.Series(results(*run()))

def sweep_ethanol_route():
    rows = []
    for r in cfg.S['ethanol_exmill_INR_L']:
        rows.append(dict(route=r, **results(*run(ethanol_route=r))))
    return pd.DataFrame(rows)

def sweep_h2_route():
    rows = []
    for r in cfg.S['h2_INR_kg']:
        rows.append(dict(route=r, **results(*run(h2_route=r))))
    return pd.DataFrame(rows)

def sweep_scale(KLPD=(50, 100, 150, 200, 300, 400)):
    rows = []
    for k in KLPD:
        rows.append(dict(KLPD=k, **results(*run(ethanol_KLPD=k))))
    return pd.DataFrame(rows)

def sweep_T_P(Ts=(373.15, 383.15, 393.15, 403.15, 413.15), Ps=(25e5, 35e5, 45e5)):
    """NEW result (impossible in the paper): selectivity and MJSP as OUTPUTS of reactor T and P."""
    rows = []
    for T, P in itertools.product(Ts, Ps):
        r = results(*run(**{'olig.T': T, 'olig.P': P}))
        rows.append(dict(T_K=T, P_bar=P / 1e5, **r))
    return pd.DataFrame(rows)

def sweep_deactivation(kds=(0.0005, 0.001, 0.002, 0.005, 0.01), cycles=(30, 60, 120)):
    rows = []
    for kd, c in itertools.product(kds, cycles):
        rows.append(dict(kd=kd, cycle_days=c, **results(*run(**{'olig.kd': kd, 'olig.cycle_days': c}))))
    return pd.DataFrame(rows)


In [ ]:
%%writefile india_atj/uncertainty.py
"""
uncertainty.py -- Monte Carlo (Latin hypercube) on the India model. Depends on: system.py, settings.py, analysis.py.

Method = the paper's (BioSTEAM Model, LHS, Spearman), with the paper-notebook bugs fixed:
  * every price setter writes `s[...]` then calls apply_prices() -> change really reaches the streams
  * naphtha setter writes RN (paper wrote RD); catalyst-LIFETIME setter writes lifetime (paper wrote the price)
  * only distributions that are NOT backed by data are flagged: 'PLACEHOLDER' ranges must be replaced by cited ranges
kind='coupled' = changes the process -> flowsheet re-simulated; kind='isolated' = economics only -> cheap.
"""
import numpy as np, pandas as pd
import chaospy as shape
import biosteam as bst
from .system import apply_prices
from . import settings as cfg

def build_model(sys, tea, F, s, MJSP_stream='SAF'):
    model = bst.Model(sys)
    st, U = F.stream, F.unit
    P = model.parameter

    def add(name, setter, element, dist, kind, units=''):
        P(setter, element=element, name=name, distribution=dist, kind=kind, units=units)

    # ---------- process (coupled) ----------
    add('Oligomerization T', lambda v: setattr(U.R202, 'T', v), U.R202, shape.Triangle(373.15, 393.15, 413.15), 'coupled', 'K')    # Heveling 1998 +-20 K (PLACEHOLDER range)
    add('Oligomerization P', lambda v: setattr(U.R202, 'P', v), U.R202, shape.Triangle(30e5, 35e5, 40e5), 'coupled', 'Pa')
    add('ASF dE (Eterm-Eprop)', lambda v: setattr(U.R202, 'dE', v), U.R202, shape.Uniform(5e3, 30e3), 'coupled', 'J/mol')       # PLACEHOLDER: fit to data
    add('Olig. conversion X0', lambda v: setattr(U.R202, 'X0', v), U.R202, shape.Triangle(0.90, 0.993, 0.999), 'coupled')
    add('Olig. deactivation kd', lambda v: setattr(U.R202, 'kd', v), U.R202, shape.Uniform(5e-4, 1e-2), 'coupled', '1/day')    # PLACEHOLDER
    add('Olig. cycle length', lambda v: setattr(U.R202, 'cycle_days', v), U.R202, shape.Uniform(30, 120), 'coupled', 'day')
    add('Olig. WHSV', lambda v: setattr(U.R202, 'whsv', v), U.R202, shape.Uniform(1.0, 2.0), 'coupled')
    add('Dehydration conversion', lambda v: setattr(U.R201, 'X', v), U.R201, shape.Triangle(0.97, 0.995, 0.999), 'coupled')
    add('Dehydration WHSV', lambda v: setattr(U.R201, 'whsv', v), U.R201, shape.Uniform(0.24, 0.36), 'coupled')              # +-20% blanket, as in paper
    add('Olig. catalyst life', lambda v: setattr(U.R202, 'cat_life_yr', v), U.R202, shape.Uniform(0.5, 2.0), 'coupled', 'yr')   # CORRECTION: sets lifetime, not price
    add('Hydrogenation catalyst life', lambda v: setattr(U.R203, 'cat_life_yr', v), U.R203, shape.Uniform(2.0, 5.0), 'coupled', 'yr')  # paper: 3-5 yr
    # ---------- economics (isolated) ----------
    def sset(key, sub=None):
        def f(v):
            (s[key].__setitem__(sub, v) if sub else s.__setitem__(key, v)); apply_prices(F, s)
        return f
    route = s['ethanol_route']; base = s['ethanol_exmill_INR_L'][route]
    add('Ethanol ex-mill price', sset('ethanol_exmill_INR_L', route), st.Ethanol_In, shape.Triangle(0.9 * base, base, 1.15 * base), 'isolated', 'INR/L')
    h2 = s['h2_INR_kg'][s['h2_route']]
    add('Hydrogen delivered price', sset('h2_INR_kg', s['h2_route']), st.Hydrogen_In, shape.Triangle(0.6 * h2, h2, 1.3 * h2), 'isolated', 'INR/kg')
    add('Naphtha (RN) price', sset('rn_USD_kg'), st.RN, shape.Uniform(0.57, 0.85), 'isolated', 'USD/kg')                      # +-20%
    add('Diesel (RD) price', sset('rd_USD_kg'), st.RD, shape.Uniform(1.51, 2.27), 'isolated', 'USD/kg')
    add('Electricity tariff', sset('elec_INR_kWh'), st.Ethanol_In, shape.Uniform(6.0, 9.0), 'isolated', 'INR/kWh')
    add('INR/USD', sset('fx'), st.Ethanol_In, shape.Triangle(90, 95.5, 102), 'isolated')
    add('Location factor', sset('location_factor'), st.Ethanol_In, shape.Uniform(0.8, 1.1), 'isolated')
    add('Loan interest', sset('finance_interest'), st.Ethanol_In, shape.Triangle(0.085, 0.095, 0.115), 'isolated')
    add('Labor cost', sset('labor_USD_yr'), st.Ethanol_In, shape.Uniform(0.8e6, 2.5e6), 'isolated', 'USD/yr')
    for k in ('dehyd', 'olig', 'hydgn'):
        base_c = s['cat_price_USD_kg'][k]
        add(f'{k} catalyst price', sset('cat_price_USD_kg', k), st['Cat_' + k], shape.Uniform(0.5 * base_c, 1.5 * base_c), 'isolated', 'USD/kg')   # paper: 50% variation

    @model.metric(name='MJSP', units='USD/gal')
    def mjsp():
        return tea.solve_price(st.SAF) * st.SAF.rho / 264.172
    @model.metric(name='MJSP', units='INR/L')
    def mjsp_inr():
        return tea.solve_price(st.SAF) * st.SAF.rho / 1000 * s['fx']
    @model.metric(name='SAF yield', units='MM gal/yr')
    def saf_yield():
        return st.SAF.F_vol * 264.172 * tea.operating_hours / 1e6
    return model

def run_mc(N=300, seed=1234, **override):
    from .analysis import run
    sys, tea, F, s = run(**override)
    model = build_model(sys, tea, F, s)
    samples = model.sample(N, rule='L', seed=seed)
    model.load_samples(samples)
    model.evaluate()
    return model

def summarize(model):
    """Percentiles of MJSP in INR/L."""
    col = [c for c in model.table.columns if 'MJSP' in str(c) and 'INR' in str(c)][0]
    y = model.table[col]
    return pd.Series({'P5': y.quantile(.05), 'P25': y.quantile(.25), 'P50': y.median(),
                      'P75': y.quantile(.75), 'P95': y.quantile(.95)})


## 1. Baseline

In [ ]:
import importlib, pandas as pd, matplotlib.pyplot as plt
from india_atj import analysis as A, uncertainty as U
base = A.baseline(); print(base.round(3))

## 2. Scenario sweeps

In [ ]:
er = A.sweep_ethanol_route(); print(er[['route','MJSP_USD_gal','MJSP_INR_L','SAF_MMgal_yr']].round(2))
hr = A.sweep_h2_route(); print(hr[['route','MJSP_USD_gal','MJSP_INR_L','H2_kg_per_tonne_fuel']].round(2))

In [ ]:
sc = A.sweep_scale(); print(sc[['KLPD','MJSP_USD_gal','SAF_MMgal_yr','FCI_MM_USD']].round(2))
plt.plot(sc.KLPD, sc.MJSP_USD_gal, 'o-'); plt.xlabel('Ethanol feed [KLPD]'); plt.ylabel('MJSP [USD/gal]'); plt.show()

In [ ]:
tp = A.sweep_T_P(); print(tp[['T_K','P_bar','alpha','SAF_frac_of_fuel','MJSP_USD_gal']].round(3))
for P,g in tp.groupby('P_bar'): plt.plot(g.T_K, g.MJSP_USD_gal, 'o-', label=f'{P:.0f} bar')
plt.xlabel('Oligomerization T [K]'); plt.ylabel('MJSP [USD/gal]'); plt.legend(); plt.show()

In [ ]:
dk = A.sweep_deactivation(); print(dk[['kd','cycle_days','olig_X_avg','MJSP_USD_gal']].round(3))

## 3. Monte Carlo (~2 s per sample; N=300 takes ~10 min)

In [ ]:
m = U.run_mc(N=300)
print(U.summarize(m).round(2))
rho, _ = m.spearman_r()
r = rho.iloc[:, [i for i,c in enumerate(rho.columns) if 'INR' in str(c)][0]]
r.reindex(r.abs().sort_values().index).plot.barh(figsize=(7,8)); plt.xlabel('Spearman rho vs MJSP [INR/L]'); plt.tight_layout(); plt.show()